## Setup

In [34]:
import os

# Define the base directory within Google Drive for your project data
BASE_DIR = '../data/raw'

os.makedirs(BASE_DIR, exist_ok=True)

print(f"Permanent 'data' folder created at: {BASE_DIR}")

for folder in os.listdir(BASE_DIR):
    print(folder)

Permanent 'data' folder created at: ../data/raw
N2_poti_TP_long_r1_780629
N1_tupi_none_short_r1_780637
N4_poti_TP_long_r1_780632
N4_poti_TP_short_r1_780631
N2_poti_PP_short_r1_780628
N2_poti_PP_long_r1_780630
N4_poti_PP_long_r1_780638
N4_poti_PP_short_r1_780634
N2_poti_TP_short_r1_780627
N1_tupi_none_long_r1_780636


### Create CSV

In [35]:
import os
import json
import re
from pathlib import Path
import pandas as pd


# =========================================================
# BASE PATH
# =========================================================

BASE_DIR = Path("../data/raw")

print(f"Using data directory: {BASE_DIR.resolve()}")

# =========================================================
# HELPERS
# =========================================================

def safe_get(d, *keys):
    for k in keys:
        if not isinstance(d, dict):
            return None
        d = d.get(k)
    return d


def load_json(path):
    try:
        with open(path, "r", encoding="utf-8") as f:
            return json.load(f)
    except Exception:
        return None


def load_jsonl_normalized(path):
    try:
        df = pd.read_json(path, lines=True)

        if df.empty:
            return pd.DataFrame()

        dfs = []

        if "metadata" in df.columns:
            dfs.append(pd.json_normalize(df["metadata"]))

        if "metrics" in df.columns:
            dfs.append(pd.json_normalize(df["metrics"]))

        if not dfs:
            return pd.DataFrame()

        return pd.concat(dfs, axis=1)

    except Exception as e:
        print(f"Error reading JSONL: {path}\n{e}")
        return pd.DataFrame()


def extract_machine_name(log_text):
    if not log_text:
        return "Unknown"

    match = re.search(r"Node\(s\)\s*:\s*([^\n]+)", log_text)
    if match:
        return match.group(1).strip()

    match = re.search(r"SubmitHost\s*:\s*([^\n]+)", log_text)
    if match:
        return match.group(1).strip()

    return "Unknown"


def detect_parallelism(name):
    """
    Infer experiment type from folder name.
    Checks for exact segment match to avoid false positives.
    """
    parts = name.upper().split("_")

    if "TP" in parts:
        return "TP"

    if "PP" in parts:
        return "PP"

    return "Single GPU"


def format_experiment_name(name):
    if not isinstance(name, str):
        return str(name)

    parts = name.split("_")

    try:
        node_part = parts[0]
        machine   = parts[1].upper()
        strategy  = parts[2]
        workload  = parts[3]

        n_nodes = int(node_part.replace("N", ""))

        strategy_map = {
            "none": "Single GPU",
            "TP":   "Tensor Parallel",
            "PP":   "Pipeline Parallel",
        }

        strategy_name = strategy_map.get(strategy, strategy)
        workload_name = workload.capitalize()

        return (
            f"{machine} ({n_nodes} node{'s' if n_nodes > 1 else ''})\n"
            f"{strategy_name}\n"
            f"{workload_name}"
        )

    except Exception:
        return name

Using data directory: /home/rafael/programacao/comp-sys-perf-analysis/data/raw


In [36]:
# =========================================================
# FILE DISCOVERY
# =========================================================

def find_experiment_files(experiment_path):
    experiment_path = Path(experiment_path)

    files = {
        "profile":        None,
        "jsonl":          None,
        "telemetry":      [],
        "server_metrics": None,
        "inputs":         None,
        "err_logs":       [],
        "out_logs":       [],
    }

    for root, _, filenames in os.walk(experiment_path):
        root = Path(root)

        for file in filenames:
            full_path = root / file

            if file == "profile_export_aiperf.json":
                files["profile"] = full_path

            elif file == "profile_export.jsonl":
                files["jsonl"] = full_path

            elif file == "server_metrics_export.json":
                files["server_metrics"] = full_path

            elif file == "inputs.json":
                files["inputs"] = full_path

            elif file == "telemetry.csv":
                files["telemetry"].append(full_path)

            elif file.endswith(".err"):
                files["err_logs"].append(full_path)

            elif file.endswith(".out"):
                files["out_logs"].append(full_path)

    return files


# =========================================================
# EXPERIMENT LOADER
# =========================================================

def load_experiment(experiment_path):
    experiment_path = Path(experiment_path)
    experiment_name = experiment_path.name

    print(f"Processing: {experiment_name}")

    files = find_experiment_files(experiment_path)

    if files["profile"] is None:
        print(f"  Skipping: no profile_export_aiperf.json")
        return None

    profile        = load_json(files["profile"])
    jsonl_df       = load_jsonl_normalized(files["jsonl"]) if files["jsonl"] else pd.DataFrame()
    server_metrics = load_json(files["server_metrics"])    if files["server_metrics"] else None
    inputs         = load_json(files["inputs"])            if files["inputs"] else None

    # --- Telemetry ---
    telemetry_data = {}
    for telemetry_file in files["telemetry"]:
        try:
            tdf = pd.read_csv(telemetry_file)
            tdf.columns = [c.strip() for c in tdf.columns]
            telemetry_data[telemetry_file.parent.name] = tdf
        except Exception as e:
            print(f"  Error loading telemetry: {telemetry_file}\n  {e}")

    # --- Logs ---
    err_logs, out_logs = {}, {}

    for f in files["err_logs"]:
        try:
            err_logs[f.name] = f.read_text(encoding="utf-8", errors="ignore")
        except Exception:
            pass

    for f in files["out_logs"]:
        try:
            out_logs[f.name] = f.read_text(encoding="utf-8", errors="ignore")
        except Exception:
            pass

    # --- Machine name ---
    machine_name = "Unknown"
    if out_logs:
        machine_name = extract_machine_name(next(iter(out_logs.values())))

    # --- Parallelism (single source of truth) ---
    parallelism = detect_parallelism(experiment_name)

    print(f"  parallelism={parallelism!r}  machine={machine_name!r}")

    return {
        "experiment_name": experiment_name,
        "pretty_name":     format_experiment_name(experiment_name),
        "parallelism":     parallelism,
        "machine_name":    machine_name,

        "profile":        profile,
        "jsonl":          jsonl_df,
        "telemetry":      telemetry_data,
        "server_metrics": server_metrics,
        "inputs":         inputs,

        "err_logs": err_logs,
        "out_logs": out_logs,
    }

In [37]:
import os
import json
import re
from pathlib import Path
import pandas as pd

# =========================================================
# TIME METRICS EXTRACTION
# =========================================================

def extract_metrics(exp):
    profile  = exp["profile"]
    df_jsonl = exp["jsonl"]

    if not profile:
        return None

    def m(name, field):
        return safe_get(profile, name, field)

    parallelism      = exp["parallelism"]
    is_communication = parallelism != "Single GPU"

    metrics = {
        # --- Experiment info ---
        "experiment_name": exp["experiment_name"],
        "display_name":    exp["pretty_name"],
        "parallelism":     parallelism,
        "machine_name":    exp["machine_name"],
        "is_communication": is_communication,

        # --- Main metrics ---
        "request_throughput_avg":       m("request_throughput",               "avg"),
        "request_latency_avg_ms":       m("request_latency",                  "avg"),
        "request_latency_std_ms":       m("request_latency",                  "std"),
        "time_to_first_token_avg_ms":   m("time_to_first_token",              "avg"),
        "time_to_first_token_std_ms":   m("time_to_first_token",              "std"),
        "inter_token_latency_avg_ms":   m("inter_token_latency",              "avg"),
        "inter_token_latency_std_ms":   m("inter_token_latency",              "std"),
        "time_to_second_token_avg_ms":  m("time_to_second_token",             "avg"),
        "time_to_second_token_std_ms":  m("time_to_second_token",             "std"),
        "output_token_throughput_avg":  m("output_token_throughput_per_user", "avg"),
        "output_token_throughput_std":  m("output_token_throughput_per_user", "std"),
    }

    # --- Percentiles ---
    if not df_jsonl.empty:

        def add_percentiles(col_name, prefix):
            if col_name not in df_jsonl.columns:
                return

            series = pd.to_numeric(
                df_jsonl[col_name],
                errors="coerce"
            ).dropna()

            if series.empty:
                return

            metrics[f"{prefix}_p50"] = series.quantile(0.50)
            metrics[f"{prefix}_p90"] = series.quantile(0.90)
            metrics[f"{prefix}_p99"] = series.quantile(0.99)

        add_percentiles(
            "request_latency.value",
            "request_latency_ms"
        )

        add_percentiles(
            "time_to_first_token.value",
            "ttft_ms"
        )

        add_percentiles(
            "time_to_second_token.value",
            "ttst_ms"
        )

        add_percentiles(
            "inter_token_latency.value",
            "inter_token_latency_ms"
        )

    # --- Token lengths ---
    if "input_sequence_length.value" in df_jsonl.columns:
        metrics["input_sequence_length_avg"] = (
            df_jsonl["input_sequence_length.value"].mean()
        )

        metrics["input_sequence_length_std"] = (
            df_jsonl["input_sequence_length.value"].std()
        )

    if "output_sequence_length.value" in df_jsonl.columns:
        metrics["output_sequence_length_avg"] = (
            df_jsonl["output_sequence_length.value"].mean()
        )

        metrics["output_sequence_length_std"] = (
            df_jsonl["output_sequence_length.value"].std()
        )

    # --- Little's Law ---
    throughput = metrics["request_throughput_avg"]
    latency    = metrics["request_latency_avg_ms"]

    metrics["avg_concurrency"] = (
        throughput * latency / 1000.0
        if throughput and latency else None
    )

    return metrics

In [38]:
# =========================================================
# GPU TELEMETRY EXTRACTION
# =========================================================

def extract_gpu_metrics(exp):

    telemetry = exp["telemetry"]

    rows = []

    for node_name, df in telemetry.items():

        if df.empty:
            continue

        row = {
            "experiment_name": exp["experiment_name"],
            "display_name":    exp["pretty_name"],
            "parallelism":     exp["parallelism"],
            "machine_name":    exp["machine_name"],
            "node_name":       node_name,
        }

        # -------------------------------------------------
        # GPU UTILIZATION
        # -------------------------------------------------

        if "utilization.gpu [%]" in df.columns:
            series = pd.to_numeric(
                df["utilization.gpu [%]"],
                errors="coerce"
            ).dropna()

            if not series.empty:
                row["gpu_util_avg"] = series.mean()
                row["gpu_util_std"] = series.std()
                row["gpu_util_p50"] = series.quantile(0.50)
                row["gpu_util_p90"] = series.quantile(0.90)
                row["gpu_util_p99"] = series.quantile(0.99)

        # -------------------------------------------------
        # MEMORY
        # -------------------------------------------------

        if "memory.used [MiB]" in df.columns:
            series = pd.to_numeric(
                df["memory.used [MiB]"],
                errors="coerce"
            ).dropna()

            if not series.empty:
                row["memory_used_avg_mib"] = series.mean()
                row["memory_used_std_mib"] = series.std()
                row["memory_used_p90_mib"] = series.quantile(0.90)

        # -------------------------------------------------
        # POWER
        # -------------------------------------------------

        if "power.draw [W]" in df.columns:
            series = pd.to_numeric(
                df["power.draw [W]"],
                errors="coerce"
            ).dropna()

            if not series.empty:
                row["power_draw_avg_w"] = series.mean()
                row["power_draw_std_w"] = series.std()
                row["power_draw_p90_w"] = series.quantile(0.90)

        # -------------------------------------------------
        # TEMPERATURE
        # -------------------------------------------------

        if "temperature.gpu" in df.columns:
            series = pd.to_numeric(
                df["temperature.gpu"],
                errors="coerce"
            ).dropna()

            if not series.empty:
                row["temperature_avg_c"] = series.mean()
                row["temperature_std_c"] = series.std()
                row["temperature_p90_c"] = series.quantile(0.90)

        rows.append(row)

    return rows

In [39]:
# =========================================================
# GPU TIMESERIES EXTRACTION
# =========================================================

def extract_gpu_timeseries(exp):

    telemetry = exp["telemetry"]

    rows = []

    for node_name, df in telemetry.items():

        if df.empty:
            continue

        df = df.copy()

        df.columns = [c.strip() for c in df.columns]

        # -------------------------------------------------
        # TIMESTAMP
        # -------------------------------------------------

        if "timestamp" in df.columns:
            df["timestamp"] = pd.to_datetime(
                df["timestamp"],
                errors="coerce"
            )

        # -------------------------------------------------
        # ITERATE ROWS
        # -------------------------------------------------

        for _, row_df in df.iterrows():

            row = {
                "experiment_name": exp["experiment_name"],
                "display_name":    exp["pretty_name"],
                "parallelism":     exp["parallelism"],
                "machine_name":    exp["machine_name"],
                "node_name":       node_name,
            }

            # -------------------------------------------------
            # TIMESTAMP
            # -------------------------------------------------

            if "timestamp" in df.columns:
                row["timestamp"] = row_df.get("timestamp")

            # -------------------------------------------------
            # GPU UTILIZATION
            # -------------------------------------------------

            if "utilization.gpu [%]" in df.columns:
                row["gpu_utilization"] = pd.to_numeric(
                    row_df.get("utilization.gpu [%]"),
                    errors="coerce"
                )

            # -------------------------------------------------
            # MEMORY
            # -------------------------------------------------

            if "memory.used [MiB]" in df.columns:
                row["memory_used_mib"] = pd.to_numeric(
                    row_df.get("memory.used [MiB]"),
                    errors="coerce"
                )

            # -------------------------------------------------
            # POWER
            # -------------------------------------------------

            if "power.draw [W]" in df.columns:
                row["power_draw_w"] = pd.to_numeric(
                    row_df.get("power.draw [W]"),
                    errors="coerce"
                )

            # -------------------------------------------------
            # TEMPERATURE
            # -------------------------------------------------

            if "temperature.gpu" in df.columns:
                row["temperature_gpu_c"] = pd.to_numeric(
                    row_df.get("temperature.gpu"),
                    errors="coerce"
                )

            rows.append(row)

    return rows

In [40]:
# =========================================================
# LOAD ALL EXPERIMENTS
# =========================================================

experiment_paths = sorted([
    p for p in BASE_DIR.iterdir()
    if p.is_dir()
])

print("\nFound experiments:")

for p in experiment_paths:
    print(f"  - {p.name}")

results         = []
gpu_results     = []
gpu_timeseries  = []

loaded_experiments = []

for exp_path in experiment_paths:

    exp = load_experiment(exp_path)

    if exp is None:
        continue

    loaded_experiments.append(exp)

    # -----------------------------------------------------
    # MAIN METRICS
    # -----------------------------------------------------

    metrics = extract_metrics(exp)

    if metrics:
        results.append(metrics)

    # -----------------------------------------------------
    # GPU AGGREGATED METRICS
    # -----------------------------------------------------

    gpu_metrics = extract_gpu_metrics(exp)

    if gpu_metrics:
        gpu_results.extend(gpu_metrics)

    # -----------------------------------------------------
    # GPU TIMESERIES
    # -----------------------------------------------------

    gpu_ts = extract_gpu_timeseries(exp)

    if gpu_ts:
        gpu_timeseries.extend(gpu_ts)


Found experiments:
  - N1_tupi_none_long_r1_780636
  - N1_tupi_none_short_r1_780637
  - N2_poti_PP_long_r1_780630
  - N2_poti_PP_short_r1_780628
  - N2_poti_TP_long_r1_780629
  - N2_poti_TP_short_r1_780627
  - N4_poti_PP_long_r1_780638
  - N4_poti_PP_short_r1_780634
  - N4_poti_TP_long_r1_780632
  - N4_poti_TP_short_r1_780631
Processing: N1_tupi_none_long_r1_780636
  parallelism='Single GPU'  machine='tupi3'
Processing: N1_tupi_none_short_r1_780637
  parallelism='Single GPU'  machine='tupi3'
Processing: N2_poti_PP_long_r1_780630
  parallelism='PP'  machine='poti[3-4]'
Processing: N2_poti_PP_short_r1_780628
  parallelism='PP'  machine='poti[3-4]'
Processing: N2_poti_TP_long_r1_780629
  parallelism='TP'  machine='poti[1-2]'
Processing: N2_poti_TP_short_r1_780627
  parallelism='TP'  machine='poti[1-2]'
Processing: N4_poti_PP_long_r1_780638
  parallelism='PP'  machine='poti[1-4]'
Processing: N4_poti_PP_short_r1_780634
  parallelism='PP'  machine='poti[1-4]'
Processing: N4_poti_TP_long_r1_

In [41]:
# =========================================================
# MAIN DATAFRAME
# =========================================================

df = pd.DataFrame(results)

if not df.empty:

    ordering = {
        "Single GPU": 0,
        "TP": 1,
        "PP": 2
    }

    df["_ord"] = (
        df["parallelism"]
        .map(ordering)
        .fillna(999)
    )

    df = (
        df
        .sort_values(["_ord", "experiment_name"])
        .drop(columns=["_ord"])
    )

    df["is_communication"] = (
        df["is_communication"]
        .astype(bool)
    )

    print("\nLoaded experiments:")
    print(df[
        [
            "experiment_name",
            "parallelism",
            "is_communication"
        ]
    ])

    # -----------------------------------------------------
    # SAVE MAIN CSV
    # -----------------------------------------------------

    output_dir = Path("../data/processed")
    output_dir.mkdir(parents=True, exist_ok=True)

    main_csv = output_dir / "summary.csv"

    df.to_csv(main_csv, index=False)

    print(f"\nSaved main metrics CSV:")
    print(main_csv.resolve())

else:
    print("No valid experiments found.")

# =========================================================
# GPU DATAFRAME
# =========================================================

gpu_df = pd.DataFrame(gpu_results)

if not gpu_df.empty:

    gpu_csv = output_dir / "gpu_summary.csv"

    gpu_df.to_csv(gpu_csv, index=False)

    print(f"\nSaved GPU metrics CSV:")
    print(gpu_csv.resolve())

# =========================================================
# GPU TIMESERIES DATAFRAME
# =========================================================

gpu_ts_df = pd.DataFrame(gpu_timeseries)

if not gpu_ts_df.empty:

    gpu_ts_df = gpu_ts_df.sort_values(
        [
            "experiment_name",
            "node_name",
            "timestamp"
        ]
    )

    gpu_ts_csv = output_dir / "gpu_timeseries.csv"

    gpu_ts_df.to_csv(gpu_ts_csv, index=False)

    print(f"\nSaved GPU timeseries CSV:")
    print(gpu_ts_csv.resolve())


Loaded experiments:
                experiment_name parallelism  is_communication
0   N1_tupi_none_long_r1_780636  Single GPU             False
1  N1_tupi_none_short_r1_780637  Single GPU             False
4     N2_poti_TP_long_r1_780629          TP              True
5    N2_poti_TP_short_r1_780627          TP              True
8     N4_poti_TP_long_r1_780632          TP              True
9    N4_poti_TP_short_r1_780631          TP              True
2     N2_poti_PP_long_r1_780630          PP              True
3    N2_poti_PP_short_r1_780628          PP              True
6     N4_poti_PP_long_r1_780638          PP              True
7    N4_poti_PP_short_r1_780634          PP              True

Saved main metrics CSV:
/home/rafael/programacao/comp-sys-perf-analysis/data/processed/summary.csv

Saved GPU metrics CSV:
/home/rafael/programacao/comp-sys-perf-analysis/data/processed/gpu_summary.csv

Saved GPU timeseries CSV:
/home/rafael/programacao/comp-sys-perf-analysis/data/processed/gpu_

### Create Summarised CSV

In [42]:
df.to_csv("../data/processed/summary.csv", index=False)